# NLP Resume-Job Matching

In [ ]:
import re
import json
import time
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

sys.path.insert(0, str(Path.cwd()))
from project_paths import CLEANED_DIR

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

CONFIG = {
    "out_root": CLEANED_DIR,
    "top_k": 10, # top job matches kept per candidate
    "tfidf_max_features": 40000,
    "tfidf_ngram": (1, 2),
    "tfidf_min_df": 5,
    "tfidf_max_df": 0.85,
    "w_text": 0.7, # weight: text similarity
    "w_skill": 0.3, # weight: skill overlap
    "batch_size": 100, # resumes processed per batch
}


def pipe_to_set(v):
    """Convert a pipe-joined string (from the cleaned CSVs) back to a set."""
    if not isinstance(v, str) or not v.strip():
        return set()
    return {s for s in v.split("|") if s}


# load data
resume_master = pd.read_csv(CONFIG["out_root"] / "master" / "resume_master.csv")
job_master = pd.read_csv(
    CONFIG["out_root"] / "master" / "job_master.csv",
    usecols=["job_id", "title", "company_name", "experience_required",
             "description_clean", "required_skills"],
)

# drop jobs/resumes without usable text
job_master = job_master[job_master["description_clean"].notna()].reset_index(drop=True)
resume_master = resume_master[resume_master["resume_clean"].notna()].reset_index(drop=True)

resume_master["skill_set"] = resume_master["extracted_skills"].map(pipe_to_set)

print("resumes:", len(resume_master), "| jobs:", len(job_master))
print("resume cols:", list(resume_master.columns))

## 1. Build TF-IDF 

In [ ]:
t0 = time.time()

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=CONFIG["tfidf_ngram"],
    max_features=CONFIG["tfidf_max_features"],
    min_df=CONFIG["tfidf_min_df"],
    max_df=CONFIG["tfidf_max_df"],
    sublinear_tf=True,
)

corpus = pd.concat([resume_master["resume_clean"], job_master["description_clean"]],
                   ignore_index=True).astype(str)
vectorizer.fit(corpus)

R = vectorizer.transform(resume_master["resume_clean"].astype(str))
J = vectorizer.transform(job_master["description_clean"].astype(str))

print(f"TF-IDF fitted in {time.time()-t0:.1f}s | vocab={len(vectorizer.vocabulary_):,}")
print("R shape:", R.shape, "| J shape:", J.shape)

## 2. Cosine similarity -> top-K jobs per candidate

In [ ]:
t0 = time.time()
JT = J.T.tocsr() # transpose job matrix and convert to Compressed Sparse Row format for fast dot-product
job_ids = job_master["job_id"].to_numpy()
res_ids = resume_master["candidate_id"].to_numpy()
k = min(CONFIG["top_k"], J.shape[0])

rows = []
for start in range(0, R.shape[0], CONFIG["batch_size"]):
    sims = R[start:start + CONFIG["batch_size"]].dot(JT).toarray()   # (b x n_job)
    for i in range(sims.shape[0]):
        sim = sims[i]
        top = np.argpartition(-sim, k - 1)[:k]
        top = top[np.argsort(-sim[top])]
        cand = res_ids[start + i]
        for jidx in top:
            rows.append((cand, job_ids[jidx], float(sim[jidx])))

match = pd.DataFrame(rows, columns=["candidate_id", "job_id", "text_similarity"])
print(f"matched {match['candidate_id'].nunique():,} candidates x top-{k} "
      f"-> {len(match):,} pairs in {time.time()-t0:.1f}s")
match.head()

## 3. Skill overlap, skill gap, and the hybrid final score

In [ ]:
# Skill dictionary 
SKILL_DICT = {
    "python": r"python", "java": r"java(?!script)", "javascript": r"java\s?script|\bjs\b",
    "c++": r"c\+\+", "c#": r"c#|c sharp", ".net": r"\.net|dotnet|asp\.net",
    "sql": r"\bsql\b|mysql|postgre|t-?sql|pl/?sql", "sql_server": r"sql server|mssql",
    "r_language": r"\br programming\b|\brstudio\b", "scala": r"\bscala\b", "go_language": r"\bgolang\b",
    "html": r"\bhtml5?\b", "css": r"\bcss3?\b", "php": r"\bphp\b", "ruby": r"\bruby\b",
    "matlab": r"matlab", "sas": r"\bsas\b", "vba": r"\bvba\b", "perl": r"\bperl\b",
    "machine_learning": r"machine learning|\bml\b", "deep_learning": r"deep learning",
    "nlp": r"natural language processing|\bnlp\b", "data_analysis": r"data analysis|data analytics",
    "data_science": r"data science", "tableau": r"tableau", "power_bi": r"power\s?bi",
    "excel": r"\bexcel\b|ms excel|microsoft excel", "spark": r"\bspark\b|pyspark",
    "hadoop": r"hadoop", "tensorflow": r"tensorflow", "pytorch": r"pytorch", "pandas": r"pandas",
    "aws": r"\baws\b|amazon web services", "azure": r"\bazure\b", "gcp": r"\bgcp\b|google cloud",
    "docker": r"docker", "kubernetes": r"kubernetes|k8s", "linux": r"\blinux\b|unix",
    "git": r"\bgit\b|github|gitlab", "jenkins": r"jenkins", "ci_cd": r"ci/?cd",
    "project_management": r"project management", "agile": r"\bagile\b|scrum",
    "ms_office": r"ms office|microsoft office", "powerpoint": r"power\s?point",
    "communication": r"communication skills?", "leadership": r"leadership",
    "customer_service": r"customer service", "sales": r"\bsales\b", "marketing": r"marketing",
    "negotiation": r"negotiation", "budgeting": r"budgeting|budget management",
    "accounting": r"accounting", "quickbooks": r"quickbooks", "auditing": r"auditing|audit",
    "financial_analysis": r"financial analysis", "taxation": r"\btax(ation)?\b", "sap": r"\bsap\b",
    "payroll": r"payroll", "forecasting": r"forecasting",
    "recruitment": r"recruitment|recruiting|talent acquisition", "onboarding": r"onboarding",
    "patient_care": r"patient care", "nursing": r"\bnursing\b|registered nurse|\brn\b",
    "autocad": r"autocad", "solidworks": r"solidworks", "photoshop": r"photoshop",
    "illustrator": r"illustrator", "adobe": r"adobe", "seo": r"\bseo\b",
    "social_media": r"social media", "salesforce": r"salesforce", "crm": r"\bcrm\b",
}
SKILL_RX = {c: re.compile(p, re.IGNORECASE) for c, p in SKILL_DICT.items()}


def extract_skills(text):
    if not isinstance(text, str):
        return set()
    return {c for c, rx in SKILL_RX.items() if rx.search(text)}


# extract skills only for the jobs that actually appear in the match results
matched_ids = match["job_id"].unique()
jobs_sub = job_master[job_master["job_id"].isin(matched_ids)][["job_id", "description_clean"]].copy()
jobs_sub["job_skill_set"] = jobs_sub["description_clean"].map(extract_skills)
job_skill_lookup = dict(zip(jobs_sub["job_id"], jobs_sub["job_skill_set"]))
resume_skill_lookup = dict(zip(resume_master["candidate_id"], resume_master["skill_set"]))
print("extracted skills for", len(job_skill_lookup), "matched jobs")

In [ ]:
# Skill overlap + hybrid score + skill gap
def skill_fields(row):
    r_sk = resume_skill_lookup.get(row["candidate_id"], set())
    j_sk = job_skill_lookup.get(row["job_id"], set())
    matching = sorted(r_sk & j_sk)
    missing = sorted(j_sk - r_sk)
    overlap = len(matching) / len(j_sk) if j_sk else 0.0
    return pd.Series({
        "matching_skills": "|".join(matching),
        "missing_skills": "|".join(missing),
        "skill_overlap": round(overlap, 4),
    })


match = pd.concat([match, match.apply(skill_fields, axis=1)], axis=1)

# hybrid final score (0-100). text_similarity and skill_overlap are both 0-1.
match["final_match_score"] = (
    100 * (CONFIG["w_text"] * match["text_similarity"] + CONFIG["w_skill"] * match["skill_overlap"])
).round(1)
match["text_match_pct"] = (match["text_similarity"] * 100).round(1)
match["skill_overlap_pct"] = (match["skill_overlap"] * 100).round(1)

# re-rank per candidate by the hybrid score
match = match.sort_values(["candidate_id", "final_match_score"], ascending=[True, False])
match["rank"] = match.groupby("candidate_id").cumcount() + 1

# attach readable job + resume info
match = (
    match.merge(job_master[["job_id", "title", "company_name"]], on="job_id", how="left")
         .merge(resume_master[["candidate_id", "role_applied"]], on="candidate_id", how="left")
         .rename(columns={"title": "job_title", "role_applied": "candidate_role"})
)

match_scores = match[[
    "candidate_id", "candidate_role", "job_id", "job_title", "company_name",
    "text_match_pct", "skill_overlap_pct", "final_match_score", "rank",
    "matching_skills", "missing_skills",
]].reset_index(drop=True)

print("match_scores shape:", match_scores.shape)
match_scores.head(10)

## 4. Ranking views (both directions) + reusable functions

- `match_resume_to_jobs(candidate_id)` - top jobs for a candidate (job-seeker view).
- `rank_candidates_for_job(job_id)` - top candidates for a job (recruiter view, "Top N candidates").
- A suitability label (Highly Suitable / Suitable / Not Suitable) is derived from the final score.

In [ ]:
def suitability(score):
    if score >= 40:
        return "Highly Suitable"
    if score >= 20:
        return "Suitable"
    return "Not Suitable"


match_scores["suitability"] = match_scores["final_match_score"].map(suitability)


def match_resume_to_jobs(candidate_id, top_n=10):
    """Job-seeker view: best job matches for one candidate (uses precomputed top-K)."""
    out = match_scores[match_scores["candidate_id"] == candidate_id].head(top_n)
    return out[["rank", "job_id", "job_title", "company_name",
                "final_match_score", "skill_overlap_pct", "suitability", "missing_skills"]]


def rank_candidates_for_job(job_id, top_n=5):
    """Recruiter view: rank ALL candidates for one job (computed on demand)."""
    jrow = job_master.index[job_master["job_id"] == job_id]
    if len(jrow) == 0:
        return pd.DataFrame()
    jvec = J[jrow[0]]
    sims = R.dot(jvec.T).toarray().ravel()          # cosine vs every resume
    j_sk = job_skill_lookup.get(job_id) or extract_skills(
        job_master.loc[jrow[0], "description_clean"])
    order = np.argsort(-sims)[:top_n * 3]            # shortlist, then blend
    recs = []
    for idx in order:
        cid = res_ids[idx]
        r_sk = resume_skill_lookup.get(cid, set())
        overlap = len(r_sk & j_sk) / len(j_sk) if j_sk else 0.0
        final = round(100 * (CONFIG["w_text"] * sims[idx] + CONFIG["w_skill"] * overlap), 1)
        recs.append((cid, round(sims[idx] * 100, 1), round(overlap * 100, 1), final))
    out = (pd.DataFrame(recs, columns=["candidate_id", "text_match_pct",
                                       "skill_overlap_pct", "final_match_score"])
           .sort_values("final_match_score", ascending=False).head(top_n).reset_index(drop=True))
    out.insert(0, "rank", out.index + 1)
    out["suitability"] = out["final_match_score"].map(suitability)
    title = job_master.loc[jrow[0], "title"]
    print(f"Top {top_n} candidates for job {job_id} - '{title}':")
    return out


# -example
demo_candidate = int(match_scores["candidate_id"].iloc[0])
print("=== Job-seeker view ===")
print(match_resume_to_jobs(demo_candidate, 5).to_string(index=False))

demo_job = int(match_scores["job_id"].iloc[0])
print("\n=== Recruiter view ===")
print(rank_candidates_for_job(demo_job, 5).to_string(index=False))

## 5. Save the Match Score Table

In [ ]:
matching_dir = CONFIG["out_root"] / "matching"
matching_dir.mkdir(parents=True, exist_ok=True)

# full Match Score Table (top-K jobs per candidate)
match_scores.to_csv(matching_dir / "match_scores.csv", index=False)

# best single match per candidate (compact ranking table)
best = match_scores[match_scores["rank"] == 1].sort_values("final_match_score", ascending=False)
best.to_csv(matching_dir / "best_match_per_candidate.csv", index=False)

# docx-style view: Candidate | Job Role | Match Score
docx_view = match_scores.rename(columns={
    "candidate_id": "Candidate", "job_title": "Job Role", "final_match_score": "Match Score(%)",
})[["Candidate", "Job Role", "Match Score(%)", "rank", "suitability"]]
docx_view.head(50).to_csv(matching_dir / "match_score_table_sample.csv", index=False)

# validation + summary
assert (match_scores.groupby("candidate_id")["rank"].min() == 1).all()
summary = {
    "n_candidates_matched": int(match_scores["candidate_id"].nunique()),
    "n_pairs": int(len(match_scores)),
    "top_k": CONFIG["top_k"],
    "avg_final_match_score": round(float(match_scores["final_match_score"].mean()), 2),
    "suitability_counts": match_scores["suitability"].value_counts().to_dict(),
    "weights": {"text": CONFIG["w_text"], "skill": CONFIG["w_skill"]},
}
with open(CONFIG["out_root"] / "reports" / "matching_summary.json", "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2)

print("Saved to cleaned_data/matching/:")
print("  - match_scores.csv (", len(match_scores), "rows )")
print("  - best_match_per_candidate.csv")
print("  - match_score_table_sample.csv")
print("\nSummary:", json.dumps(summary, indent=2))
print("\nDocx-style sample:")
print(docx_view.head(8).to_string(index=False))